# FIT5196 Assessment 2 - Missing-Value Recovery and Validation

**Group:** Group018  
**Stage:** Shared foundations, missing-value recovery and concurrent Task 2 evidence

This notebook implements the common foundations needed by the dirty-data, missing-data and
outlier work. It follows the opening structure of the supplied solution template, but it is a
separate working notebook. It does **not** repair records, remove outliers, or export cleaned
CSV files.

The source files under `data/` are treated as immutable. They are loaded into separate source
DataFrames, copied for snapshot checks, and verified by SHA-256 hashes at the beginning and end.

## 0. Configuration and environment

This section keeps identifiers, paths, expected columns, tolerances and the random seed in one
place. Central configuration makes the workflow easier to reproduce and prevents later team
members from embedding different paths or thresholds in their own code.

The notebook expects to run from the assignment directory. It reads only from `data/` and does
not create cleaning outputs in this stage.

In [1]:
from pathlib import Path

GROUP_ID = "Group018"
DATA_DIR = Path("data")
RANDOM_SEED = 5196

DIRTY_INPUT = DATA_DIR / f"{GROUP_ID}_dirty_data.csv"
MISSING_INPUT = DATA_DIR / f"{GROUP_ID}_missing_data.csv"
OUTLIER_INPUT = DATA_DIR / f"{GROUP_ID}_outlier_data.csv"
WAREHOUSE_INPUT = DATA_DIR / "warehouses.csv"
MANIFEST_INPUT = DATA_DIR / "A2_manifest.json"
RESOURCE_CONTRACT_INPUT = DATA_DIR / "resource_contract.json"
VADER_LEXICON_INPUT = DATA_DIR / "resources/nltk_data/sentiment/vader_lexicon.zip"

EXPECTED_COLUMNS = [
    "order_id",
    "customer_id",
    "date",
    "nearest_warehouse",
    "shopping_cart",
    "order_price",
    "delivery_charges",
    "customer_lat",
    "customer_long",
    "coupon_discount",
    "order_total",
    "season",
    "is_expedited_delivery",
    "distance_to_nearest_warehouse",
    "latest_customer_review",
    "is_happy_customer",
]

EXPECTED_ROWS_PER_BATCH = 500
MONEY_TOLERANCE = 0.01
DISTANCE_TOLERANCE = 0.0001
EARTH_RADIUS_KM = 6371.0
NO_REVIEW_SENTINEL = "No review submitted"

### 0.1 Dependencies and frozen local resources

The assignment requires NLTK 3.9.2 and the package-local VADER lexicon. The notebook therefore
checks the installed version and the lexicon checksum before sentiment analysis. It deliberately
contains no package-install or download commands: dependencies belong in the project environment,
and execution must work offline once they are installed.

A checksum match demonstrates that the supplied resource has not been replaced or altered.

In [2]:
import hashlib
import importlib.metadata
import json
import math
import re
import sys

import nltk
import numpy as np
import pandas as pd
from nltk.sentiment.vader import SentimentIntensityAnalyzer

print(f"Python version: {sys.version.split()[0]}")
print(f"pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"NLTK version: {nltk.__version__}")

required_nltk_version = "3.9.2"
if nltk.__version__ != required_nltk_version:
    raise RuntimeError(
        f"This notebook requires NLTK {required_nltk_version}, "
        f"but NLTK {nltk.__version__} is active."
    )

Python version: 3.12.14
pandas version: 3.0.1
NumPy version: 2.3.5
NLTK version: 3.9.2


#### Resource verification helper

This cell defines the checksum helper, verifies the frozen VADER resource and configures the analyser from the supplied local files.

In [3]:
def sha256_file(file_path):
    '''Return the SHA-256 checksum of a file.

    Parameters
    ----------
    file_path : str or pathlib.Path
        Path to the file whose bytes will be hashed.
    '''
    digest = hashlib.sha256()
    with Path(file_path).open("rb") as file_handle:
        for block in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


with RESOURCE_CONTRACT_INPUT.open("r", encoding="utf-8") as file_handle:
    resource_contract = json.load(file_handle)

expected_lexicon_checksum = resource_contract["vader_lexicon_sha256"]
observed_lexicon_checksum = sha256_file(VADER_LEXICON_INPUT)

assert observed_lexicon_checksum == expected_lexicon_checksum
assert resource_contract["nltk_version"] == nltk.__version__
print("Frozen VADER lexicon checksum and NLTK version: PASS")

# Add the supplied resource root before creating the analyser.
local_nltk_data_path = str((DATA_DIR / "resources/nltk_data").resolve())
if local_nltk_data_path not in nltk.data.path:
    nltk.data.path.insert(0, local_nltk_data_path)

sentiment_analyser = SentimentIntensityAnalyzer()

Frozen VADER lexicon checksum and NLTK version: PASS


## Task 1 - Data cleansing and explanation

### 1.0 Initial investigation and shared evidence

Before diagnosing individual records, the three batches must be loaded separately and checked
against the published structure. Separate variables preserve each batch's identity and prevent
accidental merging of the required deliverables.

The initial hashes are saved so the final cell can prove that notebook execution did not alter
the raw files. Deep snapshots provide a second, in-memory check that the source DataFrames were
not modified by later analysis.

In [4]:
source_paths = {
    "dirty": DIRTY_INPUT,
    "missing": MISSING_INPUT,
    "outlier": OUTLIER_INPUT,
    "warehouses": WAREHOUSE_INPUT,
    "manifest": MANIFEST_INPUT,
    "resource_contract": RESOURCE_CONTRACT_INPUT,
    "vader_lexicon": VADER_LEXICON_INPUT,
}

initial_source_hashes = {
    source_name: sha256_file(source_path)
    for source_name, source_path in source_paths.items()
}

dirty_source = pd.read_csv(DIRTY_INPUT)
missing_source = pd.read_csv(MISSING_INPUT)
outlier_source = pd.read_csv(OUTLIER_INPUT)
warehouses_source = pd.read_csv(WAREHOUSE_INPUT)

source_batches = {
    "dirty": dirty_source,
    "missing": missing_source,
    "outlier": outlier_source,
}

# These snapshots are used only to prove that the source DataFrames remain unchanged.
source_snapshots = {
    batch_name: dataframe.copy(deep=True)
    for batch_name, dataframe in source_batches.items()
}

print("Source files loaded without writing any output files.")

Source files loaded without writing any output files.


### 1.0.1 Structure, schema, missingness and keys

These checks establish the grain and basic quality of each batch. The specification requires
500 rows and the same 16 columns in the published order. Order IDs must be unique within and
across batches because the files contain different orders, not three versions of the same data.

Missingness is reported rather than filled. At this foundation stage, the output is evidence for
later decisions and must not change any source value.

In [5]:
overview_rows = []

for batch_name, dataframe in source_batches.items():
    overview_rows.append(
        {
            "batch": batch_name,
            "rows": len(dataframe),
            "columns": len(dataframe.columns),
            "unique_order_ids": dataframe["order_id"].nunique(),
            "unique_customers": dataframe["customer_id"].nunique(),
            "duplicate_rows": int(dataframe.duplicated().sum()),
            "missing_cells": int(dataframe.isna().sum().sum()),
        }
    )

    assert len(dataframe) == EXPECTED_ROWS_PER_BATCH
    assert list(dataframe.columns) == EXPECTED_COLUMNS
    assert dataframe["order_id"].is_unique

batch_overview = pd.DataFrame(overview_rows).set_index("batch")
display(batch_overview)

combined_order_ids = pd.concat(
    [dataframe["order_id"] for dataframe in source_batches.values()],
    ignore_index=True,
)
assert len(combined_order_ids) == 1500
assert combined_order_ids.nunique() == 1500
print("Row counts, schema, and within/across-batch order ID checks: PASS")

,rows,columns,unique_order_ids,unique_customers,duplicate_rows,missing_cells
batch,,,,,,
dirty,500,16,500,317,0,0
missing,500,16,500,312,0,81
outlier,500,16,500,318,0,0


Row counts, schema, and within/across-batch order ID checks: PASS


#### Data types and missingness

This cell compares inferred data types and missing-value counts across the three batches. The tables identify fields that require later treatment without changing them.

In [6]:
dtype_summary = pd.DataFrame(
    {
        batch_name: dataframe.dtypes.astype(str)
        for batch_name, dataframe in source_batches.items()
    }
)
missingness_summary = pd.DataFrame(
    {
        batch_name: dataframe.isna().sum()
        for batch_name, dataframe in source_batches.items()
    }
)

print("Data types")
display(dtype_summary)
print("Missing cells by field")
display(missingness_summary)

Data types


,dirty,missing,outlier
order_id,str,str,str
customer_id,str,str,str
date,str,str,str
nearest_warehouse,str,str,str
shopping_cart,str,str,str
order_price,float64,float64,float64
delivery_charges,float64,float64,float64
customer_lat,float64,float64,float64
customer_long,float64,float64,float64
coupon_discount,int64,int64,int64


Missing cells by field


,dirty,missing,outlier
order_id,0,0,0
customer_id,0,0,0
date,0,0,0
nearest_warehouse,0,11,0
shopping_cart,0,0,0
order_price,0,10,0
delivery_charges,0,10,0
customer_lat,0,0,0
customer_long,0,0,0
coupon_discount,0,0,0


#### Categorical-value review

This cell reports observed categorical values and frequencies so unexpected labels, Boolean representations and missing categories remain visible.

In [7]:
categorical_columns = [
    "nearest_warehouse",
    "season",
    "is_expedited_delivery",
    "is_happy_customer",
    "coupon_discount",
]

for column_name in categorical_columns:
    category_counts = pd.DataFrame(
        {
            batch_name: dataframe[column_name].value_counts(dropna=False)
            for batch_name, dataframe in source_batches.items()
        }
    ).fillna(0).astype(int)
    print(f"Observed values for {column_name}")
    display(category_counts)

Observed values for nearest_warehouse


,dirty,missing,outlier
nearest_warehouse,,,
Baker's,3,0,0
Bakers,174,188,170
Nickolsen,3,0,0
Nickolson,82,84,98
THOMPSON WH,3,0,0
Thompson,235,217,232
NaN,0,11,0


Observed values for season


,dirty,missing,outlier
season,,,
Autumn,137,144,122
Spring,117,116,125
Summer,121,112,134
Winter,125,118,119
NaN,0,10,0


Observed values for is_expedited_delivery


,dirty,missing,outlier
is_expedited_delivery,,,
False,402,392,408
True,98,98,92
NaN,0,10,0


Observed values for is_happy_customer


,dirty,missing,outlier
is_happy_customer,,,
False,94,91,104
True,406,399,396
NaN,0,10,0


Observed values for coupon_discount


,dirty,missing,outlier
coupon_discount,,,
0,107,116,105
5,97,95,92
10,109,104,107
15,96,99,101
20,66,61,53
25,25,25,42


### 1.0.2 Shopping-cart parsing

`shopping_cart` is JSON, even though the CSV representation contains doubled quotation marks.
The CSV reader restores the JSON string, and `json.loads` safely parses it. `eval` is not used.

The parser below also validates the expected object structure: each entry must contain a product
name and a positive integer quantity.

In [8]:
def parse_shopping_cart(cart_text):
    '''Parse and validate one shopping-cart JSON string.

    Parameters
    ----------
    cart_text : str
        JSON text containing a list of product and quantity objects.
    '''
    if not isinstance(cart_text, str):
        raise TypeError("shopping_cart must be supplied as text")

    parsed_cart = json.loads(cart_text)
    if not isinstance(parsed_cart, list) or len(parsed_cart) == 0:
        raise ValueError("shopping_cart must be a non-empty JSON list")

    for item in parsed_cart:
        if not isinstance(item, dict):
            raise ValueError("Every cart entry must be a JSON object")
        if set(item) != {"product", "quantity"}:
            raise ValueError("Every cart entry needs product and quantity keys")
        if not isinstance(item["product"], str) or not item["product"].strip():
            raise ValueError("Every product must have a non-empty text name")
        if isinstance(item["quantity"], bool) or not isinstance(item["quantity"], int):
            raise ValueError("Every quantity must be an integer")
        if item["quantity"] <= 0:
            raise ValueError("Every quantity must be positive")

    return parsed_cart

#### Cart-validation results

This cell applies the cart parser to every order and summarises product coverage, cart sizes and quantity ranges. Any malformed cart causes an explicit failure.

In [9]:
cart_summary_rows = []
all_product_names = set()

for batch_name, dataframe in source_batches.items():
    parsed_carts = []
    parsing_errors = []

    for row_index, cart_text in dataframe["shopping_cart"].items():
        try:
            parsed_carts.append(parse_shopping_cart(cart_text))
        except (TypeError, ValueError, json.JSONDecodeError) as error:
            parsing_errors.append((row_index, str(error)))

    cart_lengths = [len(cart) for cart in parsed_carts]
    quantities = [item["quantity"] for cart in parsed_carts for item in cart]
    batch_products = {
        item["product"] for cart in parsed_carts for item in cart
    }
    all_product_names.update(batch_products)

    cart_summary_rows.append(
        {
            "batch": batch_name,
            "parsing_errors": len(parsing_errors),
            "distinct_products": len(batch_products),
            "minimum_cart_lines": min(cart_lengths),
            "maximum_cart_lines": max(cart_lengths),
            "minimum_quantity": min(quantities),
            "maximum_quantity": max(quantities),
        }
    )
    assert len(parsing_errors) == 0

product_names = sorted(all_product_names)
assert len(product_names) == 12

display(pd.DataFrame(cart_summary_rows).set_index("batch"))
display(pd.DataFrame({"product": product_names}))
print("All shopping carts are valid JSON with positive integer quantities: PASS")

,parsing_errors,distinct_products,minimum_cart_lines,maximum_cart_lines,minimum_quantity,maximum_quantity
batch,,,,,,
dirty,0,12,1,5,1,3
missing,0,12,1,5,1,3
outlier,0,12,1,5,1,3


,product
0,Candle Edge 392
1,Candle Edge 932
2,Candle Flux 162
3,Candle Lumen 220
4,Candle Orbit 548
5,Candle Orbit 674
6,Candle Shift 916
7,Candle Vista 110
8,Vela Core 957
9,Vela Halo 137


All shopping carts are valid JSON with positive integer quantities: PASS


### 1.0.3 Current product-price inference

Each order price creates a linear equation: the quantity of each product multiplied by its
current unit price must sum to `order_price`. The outlier batch is suitable evidence because the
specification states that only `delivery_charges` can be anomalous there; its carts and order
prices are reliable.

A reproducible 80% subset is used to infer a provisional price vector. The remaining 20% is held
out to check that the prices reproduce unseen equations. Matrix rank must equal the number of
products, otherwise the equations do not uniquely identify every price. After validation, the
final price vector is refitted using all reliable outlier-batch equations.

In [10]:
def build_cart_matrix(dataframe, product_names):
    '''Build a product-quantity matrix from an order DataFrame.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Orders containing a shopping_cart column with JSON cart text.
    product_names : sequence of str
        Ordered product names defining the matrix columns.
    '''
    product_to_column = {
        product_name: column_index
        for column_index, product_name in enumerate(product_names)
    }
    quantity_matrix = np.zeros(
        (len(dataframe), len(product_names)),
        dtype=float,
    )

    for matrix_row, cart_text in enumerate(dataframe["shopping_cart"]):
        parsed_cart = parse_shopping_cart(cart_text)
        for item in parsed_cart:
            product_name = item["product"]
            if product_name not in product_to_column:
                raise ValueError(f"Unknown product in cart: {product_name}")
            matrix_column = product_to_column[product_name]
            quantity_matrix[matrix_row, matrix_column] += item["quantity"]

    return quantity_matrix


def infer_unit_prices(dataframe, product_names):
    '''Infer unit prices from reliable cart and order-price equations.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Reliable orders with complete shopping_cart and order_price values.
    product_names : sequence of str
        Ordered product names whose prices will be estimated.
    '''
    if dataframe["order_price"].isna().any():
        raise ValueError("Price inference requires complete order_price values")

    quantity_matrix = build_cart_matrix(dataframe, product_names)
    observed_order_prices = dataframe["order_price"].to_numpy(dtype=float)
    estimated_prices, residuals, matrix_rank, singular_values = np.linalg.lstsq(
        quantity_matrix,
        observed_order_prices,
        rcond=None,
    )

    if matrix_rank != len(product_names):
        raise ValueError(
            "The cart equations do not have full rank, so prices are not uniquely identified"
        )

    rounded_prices = np.round(estimated_prices, 2)
    price_series = pd.Series(
        rounded_prices,
        index=product_names,
        name="unit_price_aud",
    )
    diagnostics = {
        "number_of_equations": len(dataframe),
        "number_of_products": len(product_names),
        "matrix_rank": int(matrix_rank),
        "least_squares_residual_sum": (
            float(residuals[0]) if len(residuals) else 0.0
        ),
        "smallest_singular_value": float(singular_values.min()),
    }
    return price_series, diagnostics

#### Held-out price validation

This cell uses a reproducible split to infer provisional prices and test them on unused cart equations. It confirms that the price vector generalises beyond its fitting rows.

In [11]:
random_generator = np.random.default_rng(RANDOM_SEED)
shuffled_positions = random_generator.permutation(len(outlier_source))
training_size = int(0.80 * len(outlier_source))

training_positions = shuffled_positions[:training_size]
validation_positions = shuffled_positions[training_size:]
price_training_data = outlier_source.iloc[training_positions].copy(deep=True)
price_validation_data = outlier_source.iloc[validation_positions].copy(deep=True)

provisional_unit_prices, training_diagnostics = infer_unit_prices(
    price_training_data,
    product_names,
)

validation_cart_matrix = build_cart_matrix(
    price_validation_data,
    product_names,
)
validation_predictions = np.round(
    validation_cart_matrix @ provisional_unit_prices.to_numpy(),
    2,
)
validation_observed = price_validation_data["order_price"].to_numpy(dtype=float)
validation_absolute_errors = np.abs(
    validation_predictions - validation_observed
)

assert training_diagnostics["matrix_rank"] == len(product_names)
assert validation_absolute_errors.max() <= MONEY_TOLERANCE

display(pd.Series(training_diagnostics, name="training_diagnostic"))
display(
    pd.Series(
        {
            "held_out_equations": len(price_validation_data),
            "maximum_absolute_error_aud": validation_absolute_errors.max(),
            "mean_absolute_error_aud": validation_absolute_errors.mean(),
        },
        name="held_out_validation",
    )
)
print("Held-out cart-equation validation: PASS")

number_of_equations           4.000000e+02
number_of_products            1.200000e+01
matrix_rank                   1.200000e+01
least_squares_residual_sum    9.571595e-22
smallest_singular_value       9.212117e+00
Name: training_diagnostic, dtype: float64

held_out_equations            100.0
maximum_absolute_error_aud      0.0
mean_absolute_error_aud         0.0
Name: held_out_validation, dtype: float64

Held-out cart-equation validation: PASS


#### Final price vector and batch comparison

This cell refits prices on all reliable equations and compares calculated order prices with each batch. It reports inconsistencies but does not repair them.

In [12]:
final_unit_prices, final_price_diagnostics = infer_unit_prices(
    outlier_source,
    product_names,
)

display(final_unit_prices.to_frame())
display(pd.Series(final_price_diagnostics, name="final_fit_diagnostic"))

price_check_rows = []
for batch_name, dataframe in source_batches.items():
    observed_mask = dataframe["order_price"].notna()
    observed_orders = dataframe.loc[observed_mask]
    cart_matrix = build_cart_matrix(observed_orders, product_names)
    calculated_prices = np.round(
        cart_matrix @ final_unit_prices.to_numpy(),
        2,
    )
    observed_prices = observed_orders["order_price"].to_numpy(dtype=float)
    absolute_errors = np.abs(calculated_prices - observed_prices)

    price_check_rows.append(
        {
            "batch": batch_name,
            "observed_order_prices": len(observed_orders),
            "matches_within_0.01": int(
                (absolute_errors <= MONEY_TOLERANCE).sum()
            ),
            "values_to_investigate_later": int(
                (absolute_errors > MONEY_TOLERANCE).sum()
            ),
            "maximum_absolute_error_aud": absolute_errors.max(),
        }
    )

display(pd.DataFrame(price_check_rows).set_index("batch"))

,unit_price_aud
Candle Edge 392,375.75
Candle Edge 932,1490.48
Candle Flux 162,1060.29
Candle Lumen 220,3529.39
Candle Orbit 548,256.07
Candle Orbit 674,324.06
Candle Shift 916,1858.77
Candle Vista 110,1816.59
Vela Core 957,271.47
Vela Halo 137,759.89


number_of_equations           5.000000e+02
number_of_products            1.200000e+01
matrix_rank                   1.200000e+01
least_squares_residual_sum    1.394952e-21
smallest_singular_value       1.038364e+01
Name: final_fit_diagnostic, dtype: float64

,observed_order_prices,matches_within_0.01,values_to_investigate_later,maximum_absolute_error_aud
batch,,,,
dirty,500,492,8,121.75
missing,490,490,0,0.00
outlier,500,500,0,0.00


### 1.0.4 Reusable calculations for dates, seasons and order arithmetic

Dates may appear in year-first or slash-formatted day/month/year forms. The parser applies the
specification explicitly rather than asking pandas to guess. Seasons are then derived from the
calendar month using Australian seasons.

Order-price calculation follows the required item-level rounding. Order total applies the coupon
to merchandise only, adds delivery afterwards, and rounds the final result to two decimals.

In [13]:
def calculate_order_price(cart, unit_prices):
    '''Calculate a cart's merchandise price using the published rounding rule.

    Parameters
    ----------
    cart : str or list of dict
        Shopping-cart JSON text or an already parsed list of cart entries.
    unit_prices : mapping
        Mapping from each product name to its current unit price in AUD.
    '''
    parsed_cart = parse_shopping_cart(cart) if isinstance(cart, str) else cart
    item_amounts = []

    for item in parsed_cart:
        product_name = item["product"]
        if product_name not in unit_prices:
            raise KeyError(f"No unit price is available for {product_name}")
        item_amount = round(
            item["quantity"] * float(unit_prices[product_name]),
            2,
        )
        item_amounts.append(item_amount)

    return round(sum(item_amounts), 2)


def parse_order_date(date_value):
    '''Parse an order date using the explicit date rules.

    Parameters
    ----------
    date_value : str or date-like value
        Source order date. Slash-formatted numeric dates are day-first unless
        they begin with a four-digit year, which remains year-first.
    '''
    date_text = str(date_value).strip()

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", date_text):
        return pd.to_datetime(date_text, format="%Y-%m-%d")
    if re.fullmatch(r"\d{4}/\d{2}/\d{2}", date_text):
        return pd.to_datetime(date_text, format="%Y/%m/%d")
    if re.fullmatch(r"\d{1,2}/\d{1,2}/\d{4}", date_text):
        return pd.to_datetime(date_text, dayfirst=True)

    return pd.to_datetime(date_text, errors="raise")


def get_australian_season(order_date):
    '''Return the Australian season for an order date.

    Parameters
    ----------
    order_date : str or date-like value
        Order date whose calendar month determines the season.
    '''
    parsed_date = parse_order_date(order_date)
    month_number = parsed_date.month

    if month_number in (12, 1, 2):
        return "Summer"
    if month_number in (3, 4, 5):
        return "Autumn"
    if month_number in (6, 7, 8):
        return "Winter"
    return "Spring"


def calculate_order_total(order_price, coupon_discount, delivery_charge):
    '''Calculate the final order total using the published formula.

    Parameters
    ----------
    order_price : float
        Merchandise price before discount and delivery, in AUD.
    coupon_discount : float
        Percentage discount applied to merchandise only.
    delivery_charge : float
        Delivery charge added after the merchandise discount, in AUD.
    '''
    discounted_fraction = 1.0 - (float(coupon_discount) / 100.0)
    return round(
        float(order_price) * discounted_fraction + float(delivery_charge),
        2,
    )

#### Date and arithmetic function tests

This cell tests mixed date formats, season boundaries, cart pricing and the published order-total example before the functions are used for imputation.

In [14]:
assert parse_order_date("19/10/2019") == pd.Timestamp("2019-10-19")
assert parse_order_date("2019/11/09") == pd.Timestamp("2019-11-09")
assert parse_order_date("Apr 24, 2019") == pd.Timestamp("2019-04-24")

season_examples = {
    "2019-01-15": "Summer",
    "2019-03-01": "Autumn",
    "2019-06-30": "Winter",
    "2019-09-01": "Spring",
    "2019-12-31": "Summer",
}
for example_date, expected_season in season_examples.items():
    assert get_australian_season(example_date) == expected_season

assert calculate_order_total(100.00, 10, 8.00) == 98.00

example_cart = '[{"product":"Candle Edge 392","quantity":2}]'
expected_example_price = round(2 * final_unit_prices["Candle Edge 392"], 2)
assert calculate_order_price(example_cart, final_unit_prices) == expected_example_price
print("Date, season, order-price and order-total function tests: PASS")

Date, season, order-price and order-total function tests: PASS


### 1.0.5 Reusable warehouse and distance calculations

Customer-to-warehouse distance is the straight-line great-circle distance calculated with the
haversine formula and an Earth radius of 6,371 km. The nearest warehouse is the warehouse with
the minimum calculated distance. Road-routing data and shortest-path algorithms are not relevant.

Calculations retain coordinate precision; reconstructed distances are rounded to four decimals
only when returned for comparison or future repair.

In [15]:
def haversine_distance(lat1, lon1, lat2, lon2):
    '''Calculate great-circle distance between two coordinates in kilometres.

    Parameters
    ----------
    lat1 : float
        Latitude of the first point in decimal degrees.
    lon1 : float
        Longitude of the first point in decimal degrees.
    lat2 : float
        Latitude of the second point in decimal degrees.
    lon2 : float
        Longitude of the second point in decimal degrees.
    '''
    latitude_1 = math.radians(float(lat1))
    longitude_1 = math.radians(float(lon1))
    latitude_2 = math.radians(float(lat2))
    longitude_2 = math.radians(float(lon2))

    latitude_difference = latitude_2 - latitude_1
    longitude_difference = longitude_2 - longitude_1

    haversine_value = (
        math.sin(latitude_difference / 2.0) ** 2
        + math.cos(latitude_1)
        * math.cos(latitude_2)
        * math.sin(longitude_difference / 2.0) ** 2
    )
    central_angle = 2.0 * math.asin(math.sqrt(haversine_value))
    return EARTH_RADIUS_KM * central_angle


def find_nearest_warehouse(customer_lat, customer_long, warehouses):
    '''Find the closest supplied warehouse to a customer coordinate.

    Parameters
    ----------
    customer_lat : float
        Customer latitude in decimal degrees.
    customer_long : float
        Customer longitude in decimal degrees.
    warehouses : pandas.DataFrame
        Reference data containing warehouse, latitude and longitude columns.
    '''
    required_warehouse_columns = {"warehouse", "latitude", "longitude"}
    if not required_warehouse_columns.issubset(warehouses.columns):
        raise ValueError("Warehouse reference data has missing required columns")

    distances = []
    for warehouse_row in warehouses.itertuples(index=False):
        distance_km = haversine_distance(
            customer_lat,
            customer_long,
            warehouse_row.latitude,
            warehouse_row.longitude,
        )
        distances.append((warehouse_row.warehouse, distance_km))

    nearest_name, nearest_distance = min(
        distances,
        key=lambda warehouse_distance: warehouse_distance[1],
    )
    return nearest_name, round(nearest_distance, 4)

#### Warehouse function tests

This cell tests zero distance at each warehouse and demonstrates the nearest-warehouse calculation on a supplied order.

In [16]:
assert haversine_distance(-37.8, 145.0, -37.8, 145.0) == 0.0

for warehouse_row in warehouses_source.itertuples(index=False):
    calculated_name, calculated_distance = find_nearest_warehouse(
        warehouse_row.latitude,
        warehouse_row.longitude,
        warehouses_source,
    )
    assert calculated_name == warehouse_row.warehouse
    assert calculated_distance == 0.0

example_order = outlier_source.iloc[0]
example_warehouse, example_distance = find_nearest_warehouse(
    example_order["customer_lat"],
    example_order["customer_long"],
    warehouses_source,
)
print(f"Example calculated warehouse: {example_warehouse}")
print(f"Example calculated distance: {example_distance:.4f} km")
print("Haversine and nearest-warehouse function tests: PASS")

Example calculated warehouse: Nickolson
Example calculated distance: 7.2270 km
Haversine and nearest-warehouse function tests: PASS


### 1.0.6 Reusable VADER happiness classification

The supplied review must be scored exactly as written. A customer is happy when the VADER
compound score is greater than or equal to 0.05. The exact text `No review submitted` must return `True` without being treated as missing.

A small dummy analyser is used to test the inclusive boundary exactly. This avoids relying on a
particular sentence to happen to receive a compound score of precisely 0.05.

In [17]:
def classify_customer_happiness(review_text, analyser):
    '''Classify prior-review happiness using the assignment's VADER rule.

    Parameters
    ----------
    review_text : str
        Supplied English review text or the exact no-review sentinel.
    analyser : nltk.sentiment.vader.SentimentIntensityAnalyzer
        Configured sentiment analyser using the frozen local lexicon.
    '''
    if review_text == NO_REVIEW_SENTINEL:
        return True
    if not isinstance(review_text, str):
        raise TypeError("latest_customer_review must be text")

    compound_score = analyser.polarity_scores(review_text)["compound"]
    return bool(compound_score >= resource_contract["compound_threshold"])


class BoundaryTestAnalyser:
    '''Return a fixed compound score for an exact threshold test.'''

    def polarity_scores(self, review_text):
        '''Return a fixed VADER-like score dictionary.

        Parameters
        ----------
        review_text : str
            Review text accepted to match the real analyser interface.
        '''
        return {"compound": 0.05}


assert classify_customer_happiness(
    NO_REVIEW_SENTINEL,
    sentiment_analyser,
) is True
assert classify_customer_happiness(
    "Boundary test",
    BoundaryTestAnalyser(),
) is True
assert classify_customer_happiness(
    "This product is excellent and reliable.",
    sentiment_analyser,
) is True
assert classify_customer_happiness(
    "This product is terrible and disappointing.",
    sentiment_analyser,
) is False
print("VADER resource, sentinel and inclusive-threshold tests: PASS")

VADER resource, sentinel and inclusive-threshold tests: PASS


### 1.0.7 Final foundation checks and stage boundary

The final checks repeat the source hashes and compare every source DataFrame with its deep
snapshot. A passing result confirms that the investigation and demonstrations did not modify the
original files or the in-memory source tables.

This notebook intentionally stops here. Later notebooks or sections may call these functions on
explicit working copies to diagnose dirty fields, recover missing values and model delivery-charge
residuals. No correction should be made merely because one dependent consistency check fails.

In [18]:
final_source_hashes = {
    source_name: sha256_file(source_path)
    for source_name, source_path in source_paths.items()
}
assert final_source_hashes == initial_source_hashes

for batch_name, source_dataframe in source_batches.items():
    pd.testing.assert_frame_equal(
        source_dataframe,
        source_snapshots[batch_name],
        check_dtype=True,
        check_exact=True,
    )

unexpected_output_names = [
    f"{GROUP_ID}_dirty_clean.csv",
    f"{GROUP_ID}_outlier_clean.csv",
    "audit_log.csv",
]
created_unexpected_outputs = [
    output_name
    for output_name in unexpected_output_names
    if Path(output_name).exists()
]
assert created_unexpected_outputs == []

final_summary = pd.Series(
    {
        "source_file_hashes_unchanged": True,
        "source_dataframes_unchanged": True,
        "unexpected_cleaning_outputs_created": False,
        "batches_checked": len(source_batches),
        "distinct_order_ids": combined_order_ids.nunique(),
        "products_identified": len(product_names),
        "price_matrix_rank": final_price_diagnostics["matrix_rank"],
        "vader_contract_verified": True,
    },
    name="shared_foundation_result",
)
display(final_summary)

source_file_hashes_unchanged            True
source_dataframes_unchanged             True
unexpected_cleaning_outputs_created    False
batches_checked                            3
distinct_order_ids                      1500
products_identified                       12
price_matrix_rank                         12
vader_contract_verified                 True
Name: shared_foundation_result, dtype: object

## 1.2 Missing data - recover omitted values

This section recovers only the empty fields in the missing-data batch. The specification
guarantees that every observed value in this batch is correct, so the workflow starts from a
deep copy and writes only to cells that were empty in the source.

Deterministic business relationships are preferred wherever they are available. The missing
expedited-delivery values are the only exception: they are inferred by comparing seasonal
delivery-model predictions under the two possible Boolean values.

In [19]:
MISSING_OUTPUT = Path(f"{GROUP_ID}_missing_clean.csv")
MISSING_SOURCE_FILENAME = MISSING_INPUT.name

missing_clean = missing_source.copy(deep=True)
original_missing_mask = missing_source.isna()
missing_cells_per_row = original_missing_mask.sum(axis=1)

missing_counts_before = original_missing_mask.sum()
missing_counts_before = missing_counts_before[missing_counts_before > 0]

assert int(original_missing_mask.sum().sum()) == 81
assert int((missing_cells_per_row > 0).sum()) == 81
assert int(missing_cells_per_row.max()) == 1

display(missing_counts_before.rename("missing_cells").to_frame())
print("The source contains 81 missing cells across 81 separate rows: PASS")

,missing_cells
nearest_warehouse,11
order_price,10
delivery_charges,10
order_total,10
season,10
is_expedited_delivery,10
distance_to_nearest_warehouse,10
is_happy_customer,10


The source contains 81 missing cells across 81 separate rows: PASS


### 1.2.1 Audit-record structure and confidence definitions

This cell prepares the missing-data portion of the Task 2 audit trail at the moment each value
is filled. Every record uses the final required column order. The original value is recorded as
`<MISSING>`, and the issue type is `missing`.

Confidence has the following meaning:

- **high:** a deterministic rule produces the value, or the model-supported choice is unambiguous;
- **medium:** evidence supports the estimate but meaningful uncertainty remains;
- **low:** evidence is weak or plausible alternatives remain unresolved.

In [20]:
AUDIT_COLUMNS = [
    "source_file",
    "record_key",
    "field_name",
    "issue_type",
    "original_value",
    "corrected_value",
    "method",
    "evidence",
    "confidence",
]

missing_change_records = []


def format_audit_value(field_name, value):
    '''Format a corrected value for the future audit CSV.

    Parameters
    ----------
    field_name : str
        Name of the field whose corrected value is being formatted.
    value : object
        Corrected Python value to represent in the audit record.
    '''
    if field_name in {"order_price", "delivery_charges", "order_total"}:
        return f"{float(value):.2f}"
    if field_name == "distance_to_nearest_warehouse":
        return f"{float(value):.4f}"
    if field_name in {"is_expedited_delivery", "is_happy_customer"}:
        return "True" if bool(value) else "False"
    return str(value)


def record_missing_imputation(
    records,
    order_id,
    field_name,
    corrected_value,
    method,
    evidence,
    confidence="high",
):
    '''Append one missing-value imputation to an audit-record list.

    Parameters
    ----------
    records : list of dict
        Mutable list that collects audit records.
    order_id : str
        Stable order identifier for the affected record.
    field_name : str
        Name of the field whose missing value was recovered.
    corrected_value : object
        Recovered value written to the cleaned working copy.
    method : str
        Short description of the rule, calculation or model used.
    evidence : str
        Traceable notebook evidence supporting the imputation.
    confidence : str, default "high"
        Confidence level using the definitions stated above.
    '''
    if confidence not in {"high", "medium", "low"}:
        raise ValueError("confidence must be high, medium or low")

    records.append(
        {
            "source_file": MISSING_SOURCE_FILENAME,
            "record_key": str(order_id),
            "field_name": field_name,
            "issue_type": "missing",
            "original_value": "<MISSING>",
            "corrected_value": format_audit_value(field_name, corrected_value),
            "method": method,
            "evidence": evidence,
            "confidence": confidence,
        }
    )

### 1.2.2 Seasonal delivery models

This cell defines the model helpers used for the missing expedited-delivery values. Each seasonal
model contains an intercept, distance, expedited-delivery status and customer happiness. Only rows
with complete observed values for all model fields are eligible, which leaves 450 guaranteed-correct
rows from the missing batch.

The helpers keep model fitting separate from validation and imputation. This makes the training
evidence reproducible and prevents the later audit checks from feeding values back into the model.

In [21]:
DELIVERY_MODEL_COLUMNS = [
    "season",
    "delivery_charges",
    "distance_to_nearest_warehouse",
    "is_expedited_delivery",
    "is_happy_customer",
]
SEASON_ORDER = ["Summer", "Autumn", "Winter", "Spring"]


def build_delivery_design_matrix(dataframe, expedited_override=None):
    '''Build the delivery-model predictor matrix for a set of orders.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Orders containing distance, expedited and happiness predictors.
    expedited_override : bool or None, default None
        Optional Boolean used for every row instead of the stored expedited value.
    '''
    row_count = len(dataframe)
    intercept = np.ones(row_count, dtype=float)
    distance = dataframe["distance_to_nearest_warehouse"].to_numpy(dtype=float)

    if expedited_override is None:
        expedited = dataframe["is_expedited_delivery"].to_numpy(dtype=float)
    else:
        expedited = np.full(row_count, float(expedited_override), dtype=float)

    happiness = dataframe["is_happy_customer"].to_numpy(dtype=float)
    return np.column_stack([intercept, distance, expedited, happiness])


def fit_seasonal_delivery_model(season_data):
    '''Fit one season's linear delivery-charge model.

    Parameters
    ----------
    season_data : pandas.DataFrame
        Complete reliable orders belonging to one Australian season.
    '''
    if season_data[DELIVERY_MODEL_COLUMNS].isna().any().any():
        raise ValueError("Seasonal model data must be complete")

    design_matrix = build_delivery_design_matrix(season_data)
    observed_charges = season_data["delivery_charges"].to_numpy(dtype=float)
    coefficients, residuals, matrix_rank, singular_values = np.linalg.lstsq(
        design_matrix,
        observed_charges,
        rcond=None,
    )

    if matrix_rank != design_matrix.shape[1]:
        raise ValueError("Seasonal delivery-model predictors do not have full rank")

    return {
        "coefficients": coefficients,
        "matrix_rank": int(matrix_rank),
        "residual_sum": float(residuals[0]) if len(residuals) else 0.0,
        "smallest_singular_value": float(singular_values.min()),
    }


def predict_delivery_charges(dataframe, fitted_model, expedited_override=None):
    '''Predict delivery charges from a fitted seasonal model.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Orders containing the required model predictors.
    fitted_model : dict
        Model information returned by fit_seasonal_delivery_model.
    expedited_override : bool or None, default None
        Optional expedited value used instead of the stored column.
    '''
    design_matrix = build_delivery_design_matrix(dataframe,expedited_override=expedited_override)
    return design_matrix @ fitted_model["coefficients"]


def evaluate_seasonal_delivery_models(dataframe, random_seed):
    '''Evaluate and refit separate delivery models for all four seasons.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Missing-data batch containing reliable complete model rows.
    random_seed : int
        Seed used to create reproducible per-season 80/20 splits.
    '''
    eligible_data = dataframe.dropna(subset=DELIVERY_MODEL_COLUMNS).copy(deep=True)
    random_generator = np.random.default_rng(random_seed)
    metric_rows = []
    refitted_models = {}

    for season_name in SEASON_ORDER:
        season_data = eligible_data.loc[
            eligible_data["season"] == season_name
        ].copy(deep=True)
        shuffled_positions = random_generator.permutation(len(season_data))
        training_size = int(0.80 * len(season_data))
        training_data = season_data.iloc[shuffled_positions[:training_size]]
        testing_data = season_data.iloc[shuffled_positions[training_size:]]

        evaluation_model = fit_seasonal_delivery_model(training_data)
        predictions = predict_delivery_charges(testing_data, evaluation_model)
        observed = testing_data["delivery_charges"].to_numpy(dtype=float)
        residuals = observed - predictions
        total_variation = np.sum((observed - observed.mean()) ** 2)
        r_squared = 1.0 - (np.sum(residuals**2) / total_variation)
        mean_absolute_error = float(np.mean(np.abs(residuals)))

        metric_rows.append(
            {
                "season": season_name,
                "eligible_rows": len(season_data),
                "training_rows": len(training_data),
                "held_out_rows": len(testing_data),
                "r_squared": r_squared,
                "mean_absolute_error_aud": mean_absolute_error,
            }
        )
        refitted_models[season_name] = fit_seasonal_delivery_model(season_data)

    metrics = pd.DataFrame(metric_rows).set_index("season")
    return metrics, refitted_models, eligible_data


def infer_expedited_delivery(order_row, seasonal_models):
    '''Infer a missing expedited-delivery value using seasonal predictions.

    Parameters
    ----------
    order_row : pandas.Series
        One order with observed season, distance, happiness and delivery charge.
    seasonal_models : dict
        Mapping from season name to its refitted delivery model.
    '''
    season_name = order_row["season"]
    model = seasonal_models[season_name]
    one_row = order_row.to_frame().T

    predicted_false = float(
        predict_delivery_charges(one_row, model, expedited_override=False)[0]
    )
    predicted_true = float(
        predict_delivery_charges(one_row, model, expedited_override=True)[0]
    )
    observed_charge = float(order_row["delivery_charges"])
    false_residual = abs(observed_charge - predicted_false)
    true_residual = abs(observed_charge - predicted_true)
    selected_value = true_residual < false_residual
    selected_residual = min(false_residual, true_residual)
    alternative_residual = max(false_residual, true_residual)

    return {
        "selected_value": bool(selected_value),
        "predicted_if_false": predicted_false,
        "predicted_if_true": predicted_true,
        "residual_if_false": false_residual,
        "residual_if_true": true_residual,
        "selected_residual": selected_residual,
        "alternative_residual": alternative_residual,
        "residual_separation": alternative_residual - selected_residual,
    }

### 1.2.3 Held-out delivery-model evaluation

This cell evaluates each season on observations that were not used to fit that evaluation model.
It reports eligible, training and held-out sample sizes together with R-squared and mean absolute
error. A clean seasonal relationship should produce R-squared of at least 0.99, but the error
measure is also reported because R-squared alone is not sufficient evidence.

After evaluation, each model is refitted on all eligible rows from its season for the final
expedited-delivery inference.

In [22]:
delivery_model_metrics, seasonal_delivery_models, delivery_model_training_data = (
    evaluate_seasonal_delivery_models(missing_source, RANDOM_SEED)
)

assert len(delivery_model_training_data) == 450
assert (delivery_model_metrics["r_squared"] >= 0.99).all()

display(delivery_model_metrics.round(6))

coefficient_rows = []
for season_name, fitted_model in seasonal_delivery_models.items():
    coefficients = fitted_model["coefficients"]
    coefficient_rows.append(
        {
            "season": season_name,
            "intercept": coefficients[0],
            "distance_coefficient": coefficients[1],
            "expedited_coefficient": coefficients[2],
            "happiness_coefficient": coefficients[3],
        }
    )

seasonal_model_coefficients = pd.DataFrame(coefficient_rows).set_index("season")
display(seasonal_model_coefficients.round(6))
print("Per-season held-out R-squared and absolute-error checks: PASS")

,eligible_rows,training_rows,held_out_rows,r_squared,mean_absolute_error_aud
season,,,,,
Summer,100,80,20,1.0,0.001974
Autumn,132,105,27,1.0,0.002608
Winter,109,87,22,1.0,0.002532
Spring,109,87,22,1.0,0.002193


,intercept,distance_coefficient,expedited_coefficient,happiness_coefficient
season,,,,
Summer,10.001157,0.699945,9.999641,-2.000917
Autumn,9.999833,0.780117,10.999774,-3.000815
Winter,14.000939,0.850047,12.998775,-4.000947
Spring,9.999771,0.729938,10.499179,-2.499145


Per-season held-out R-squared and absolute-error checks: PASS


### 1.2.4 Warehouse names and distances

This cell fills a missing warehouse name by selecting the smallest haversine distance from the
customer's coordinates. A missing distance uses the same calculation and retains four decimal
places. Only the missing target field is written, even though the helper returns both results.

In [23]:
# Impute missing nearest warehouse name
for row_index in missing_source.index[missing_source["nearest_warehouse"].isna()]:
    source_row = missing_source.loc[row_index]
    nearest_name, nearest_distance = find_nearest_warehouse(
        source_row["customer_lat"],
        source_row["customer_long"],
        warehouses_source,
    )
    missing_clean.at[row_index, "nearest_warehouse"] = nearest_name 
    record_missing_imputation(
        missing_change_records,
        source_row["order_id"],
        "nearest_warehouse",
        nearest_name,
        "Minimum haversine distance to the supplied warehouses",
        f"Section 1.2.4; calculated distance {nearest_distance:.4f} km",
    )
# Impute missing warehouse distance
for row_index in missing_source.index[
    missing_source["distance_to_nearest_warehouse"].isna()
]:
    source_row = missing_source.loc[row_index]
    nearest_name, nearest_distance = find_nearest_warehouse(
        source_row["customer_lat"],
        source_row["customer_long"],
        warehouses_source,
    )
    assert nearest_name == source_row["nearest_warehouse"]
    missing_clean.at[row_index, "distance_to_nearest_warehouse"] = nearest_distance 
    record_missing_imputation(
        missing_change_records,
        source_row["order_id"],
        "distance_to_nearest_warehouse",
        nearest_distance,
        "Minimum haversine distance using Earth radius 6,371 km",
        f"Section 1.2.4; nearest warehouse {nearest_name}",
    )

print("Warehouse names filled:", int(missing_source["nearest_warehouse"].isna().sum()))
print(
    "Warehouse distances filled:",
    int(missing_source["distance_to_nearest_warehouse"].isna().sum()),
)

Warehouse names filled: 11
Warehouse distances filled: 10


### 1.2.5 Merchandise prices

This cell calculates every missing `order_price` from its parsed shopping cart and the validated
current unit-price vector. It follows the required item-level rounding before summing the order.
The observed cart and inferred price vector are not altered.

In [24]:
for row_index in missing_source.index[missing_source["order_price"].isna()]:
    source_row = missing_source.loc[row_index]
    calculated_price = calculate_order_price(
        source_row["shopping_cart"],
        final_unit_prices,
    )
    missing_clean.at[row_index, "order_price"] = calculated_price
    record_missing_imputation(
        missing_change_records,
        source_row["order_id"],
        "order_price",
        calculated_price,
        "Parsed cart quantities multiplied by validated current unit prices",
        "Section 1.2.5; held-out price-vector validation passed",
    )

print("Order prices filled:", int(missing_source["order_price"].isna().sum()))

Order prices filled:

 10


### 1.2.6 Delivery charges and order totals

This cell uses the published arithmetic relationship in both directions. A missing delivery charge
is isolated from the observed total, price and discount. A missing total is calculated by applying
the discount to merchandise only and adding delivery afterwards.

These deterministic relationships are preferred to model estimates because they use exact values
from the same order. Reconstructed monetary values are rounded to two decimal places.

In [25]:
for row_index in missing_source.index[missing_source["delivery_charges"].isna()]:
    source_row = missing_source.loc[row_index]
    discounted_fraction = 1.0 - (float(source_row["coupon_discount"]) / 100.0)
    calculated_delivery = round(
        float(source_row["order_total"])
        - float(source_row["order_price"]) * discounted_fraction,
        2,
    )
    missing_clean.at[row_index, "delivery_charges"] = calculated_delivery

    # Verify that the calculated delivery charge is accurate
    recalculated_total = calculate_order_total(
        source_row["order_price"],
        source_row["coupon_discount"],
        calculated_delivery,
    )
    assert abs(recalculated_total - source_row["order_total"]) <= MONEY_TOLERANCE

    record_missing_imputation(
        missing_change_records,
        source_row["order_id"],
        "delivery_charges",
        calculated_delivery,
        "Rearranged published order-total equation",
        "Section 1.2.6; forward total recalculation within AUD 0.01",
    )

for row_index in missing_source.index[missing_source["order_total"].isna()]:
    source_row = missing_source.loc[row_index]
    calculated_total = calculate_order_total(
        source_row["order_price"],
        source_row["coupon_discount"],
        source_row["delivery_charges"],
    )
    missing_clean.at[row_index, "order_total"] = calculated_total
    record_missing_imputation(
        missing_change_records,
        source_row["order_id"],
        "order_total",
        calculated_total,
        "Published discount and delivery arithmetic",
        "Section 1.2.6; coupon applied to merchandise only",
    )

print(
    "Delivery charges filled:",
    int(missing_source["delivery_charges"].isna().sum()),
)
print("Order totals filled:", int(missing_source["order_total"].isna().sum()))

Delivery charges filled: 10
Order totals filled: 10


### 1.2.7 Calendar seasons

This cell derives each missing season from the parsed order date. It uses Australian calendar
seasons, so December to February is Summer, March to May is Autumn, June to August is Winter,
and September to November is Spring.

In [26]:
for row_index in missing_source.index[missing_source["season"].isna()]:
    source_row = missing_source.loc[row_index]
    calculated_season = get_australian_season(source_row["date"])
    missing_clean.at[row_index, "season"] = calculated_season
    record_missing_imputation(
        missing_change_records,
        source_row["order_id"],
        "season",
        calculated_season,
        "Australian season derived from the order-date month",
        f"Section 1.2.7; source date {source_row['date']}",
    )

print("Seasons filled:", int(missing_source["season"].isna().sum()))

Seasons filled: 10


### 1.2.8 Customer happiness

This cell applies the checksum-verified local VADER analyser to the supplied review text exactly
as written. A compound score of at least 0.05 is happy, while the exact no-review sentinel is
treated as happy without being imputed or rewritten.

In [27]:
for row_index in missing_source.index[missing_source["is_happy_customer"].isna()]:
    source_row = missing_source.loc[row_index]
    calculated_happiness = classify_customer_happiness(
        source_row["latest_customer_review"],
        sentiment_analyser,
    )
    missing_clean.at[row_index, "is_happy_customer"] = calculated_happiness
    record_missing_imputation(
        missing_change_records,
        source_row["order_id"],
        "is_happy_customer",
        calculated_happiness,
        "Frozen local VADER compound score with threshold >= 0.05",
        "Section 1.2.8; resource checksum and NLTK version verified",
    )

print(
    "Happiness values filled:",
    int(missing_source["is_happy_customer"].isna().sum()),
)

Happiness values filled: 10


### 1.2.9 Expedited-delivery status

This cell evaluates each missing Boolean twice: once as non-expedited and once as expedited. The
seasonal model predicts the delivery charge for both possibilities, and the value with the smaller
absolute residual is selected.

The diagnostic table retains both predictions and residuals. A selection receives high confidence
only when its residual is within the delivery-charge tolerance and the alternative is separated by
at least AUD 1.00. Otherwise the record remains visible with lower confidence rather than being
presented as certain.

In [28]:
expedited_diagnostic_rows = []

for row_index in missing_source.index[
    missing_source["is_expedited_delivery"].isna()
]:
    source_row = missing_source.loc[row_index]
    diagnostics = infer_expedited_delivery(
        source_row,
        seasonal_delivery_models,
    )
    selected_value = diagnostics["selected_value"]

    if (
        diagnostics["selected_residual"] <= 0.08
        and diagnostics["residual_separation"] >= 1.00
    ):
        confidence = "high"
    elif diagnostics["selected_residual"] <= 0.50:
        confidence = "medium"
    else:
        confidence = "low"

    missing_clean.at[row_index, "is_expedited_delivery"] = selected_value
    record_missing_imputation(
        missing_change_records,
        source_row["order_id"],
        "is_expedited_delivery",
        selected_value,
        "Boolean option with the smaller seasonal delivery-model residual",
        (
            "Section 1.2.9; selected residual "
            f"{diagnostics['selected_residual']:.4f}, alternative residual "
            f"{diagnostics['alternative_residual']:.4f}"
        ),
        confidence=confidence,
    )

    expedited_diagnostic_rows.append(
        {
            "order_id": source_row["order_id"],
            "season": source_row["season"],
            "observed_delivery_charge": source_row["delivery_charges"],
            "predicted_if_false": diagnostics["predicted_if_false"],
            "predicted_if_true": diagnostics["predicted_if_true"],
            "residual_if_false": diagnostics["residual_if_false"],
            "residual_if_true": diagnostics["residual_if_true"],
            "selected_value": selected_value,
            "residual_separation": diagnostics["residual_separation"],
            "confidence": confidence,
        }
    )

expedited_diagnostics = pd.DataFrame(expedited_diagnostic_rows).set_index("order_id")
display(expedited_diagnostics.round(4))
assert (expedited_diagnostics["confidence"] == "high").all()
print(
    "Expedited-delivery values filled:",
    int(missing_source["is_expedited_delivery"].isna().sum()),
)

,season,observed_delivery_charge,predicted_if_false,predicted_if_true,residual_if_false,residual_if_true,selected_value,residual_separation,confidence
order_id,,,,,,,,,
CORD000529,Autumn,9.39,9.3926,20.3924,0.0026,11.0024,False,10.9998,high
CORD000622,Autumn,21.60,10.6035,21.6032,10.9965,0.0032,True,10.9933,high
CORD000682,Autumn,15.58,15.5804,26.5802,0.0004,11.0002,False,10.9998,high
CORD000692,Summer,22.96,12.9588,22.9585,10.0012,0.0015,True,9.9996,high
CORD000735,Autumn,22.88,11.8809,22.8807,10.9991,0.0007,True,10.9984,high
CORD000743,Winter,28.96,15.9649,28.9637,12.9951,0.0037,True,12.9913,high
CORD000765,Summer,22.81,12.8144,22.8140,9.9956,0.0040,True,9.9916,high
CORD000846,Summer,9.80,9.7973,19.7969,0.0027,9.9969,False,9.9942,high
CORD000897,Autumn,12.26,12.2628,23.2626,0.0028,11.0026,False,10.9998,high


Expedited-delivery values filled: 10


### 1.2.10 Completed missing-data audit records

This cell converts the recorded changes to the final Task 2 column structure and checks that all
81 missing cells have exactly one traceable record. The table remains in the notebook so the final
workflow can concatenate it with dirty and outlier records before writing `audit_log.csv`.

In [29]:
missing_clean["is_expedited_delivery"] = missing_clean[
    "is_expedited_delivery"
].astype(bool)
missing_clean["is_happy_customer"] = missing_clean["is_happy_customer"].astype(bool)

missing_audit_records = pd.DataFrame(
    missing_change_records,
    columns=AUDIT_COLUMNS,
)

assert len(missing_audit_records) == 81
assert not missing_audit_records.duplicated(
    subset=["record_key", "field_name"]
).any()
assert set(missing_audit_records["issue_type"]) == {"missing"}
assert set(missing_audit_records["original_value"]) == {"<MISSING>"}

audit_counts_by_field = missing_audit_records["field_name"].value_counts()
expected_audit_counts = missing_counts_before.astype(int)
pd.testing.assert_series_equal(
    audit_counts_by_field.sort_index(),
    expected_audit_counts.sort_index(),
    check_names=False,
)

display(audit_counts_by_field.rename("audit_rows").to_frame())
display(missing_audit_records.head(10))
print("Missing-data audit coverage: 81 of 81 cells recorded")

,audit_rows
field_name,
nearest_warehouse,11
distance_to_nearest_warehouse,10
order_price,10
delivery_charges,10
order_total,10
season,10
is_happy_customer,10
is_expedited_delivery,10


,source_file,record_key,field_name,issue_type,original_value,corrected_value,method,evidence,confidence
0,Group018_missing_data.csv,CORD000524,nearest_warehouse,missing,<MISSING>,Thompson,Minimum haversine distance to the supplied war...,Section 1.2.4; calculated distance 3.6405 km,high
1,Group018_missing_data.csv,CORD000581,nearest_warehouse,missing,<MISSING>,Thompson,Minimum haversine distance to the supplied war...,Section 1.2.4; calculated distance 2.8326 km,high
2,Group018_missing_data.csv,CORD000659,nearest_warehouse,missing,<MISSING>,Bakers,Minimum haversine distance to the supplied war...,Section 1.2.4; calculated distance 5.5760 km,high
3,Group018_missing_data.csv,CORD000697,nearest_warehouse,missing,<MISSING>,Nickolson,Minimum haversine distance to the supplied war...,Section 1.2.4; calculated distance 5.5854 km,high
4,Group018_missing_data.csv,CORD000745,nearest_warehouse,missing,<MISSING>,Bakers,Minimum haversine distance to the supplied war...,Section 1.2.4; calculated distance 7.1010 km,high
5,Group018_missing_data.csv,CORD000773,nearest_warehouse,missing,<MISSING>,Thompson,Minimum haversine distance to the supplied war...,Section 1.2.4; calculated distance 4.6966 km,high
6,Group018_missing_data.csv,CORD000856,nearest_warehouse,missing,<MISSING>,Nickolson,Minimum haversine distance to the supplied war...,Section 1.2.4; calculated distance 1.5241 km,high
7,Group018_missing_data.csv,CORD000867,nearest_warehouse,missing,<MISSING>,Bakers,Minimum haversine distance to the supplied war...,Section 1.2.4; calculated distance 6.5436 km,high
8,Group018_missing_data.csv,CORD000901,nearest_warehouse,missing,<MISSING>,Thompson,Minimum haversine distance to the supplied war...,Section 1.2.4; calculated distance 8.0880 km,high
9,Group018_missing_data.csv,CORD000959,nearest_warehouse,missing,<MISSING>,Bakers,Minimum haversine distance to the supplied war...,Section 1.2.4; calculated distance 5.9319 km,high


Missing-data audit coverage: 81 of 81 cells recorded


### 1.2.11 Export the missing-data output

This cell verifies completeness and the required schema before writing the cleaned missing batch.
It retains all 500 order IDs, preserves the original column order, uses UTF-8 CSV encoding and
does not write a pandas index. No audit CSV is written at this stage.

In [30]:
assert missing_clean.isna().sum().sum() == 0
assert len(missing_clean) == EXPECTED_ROWS_PER_BATCH
assert list(missing_clean.columns) == EXPECTED_COLUMNS
assert missing_clean["order_id"].is_unique
assert set(missing_clean["order_id"]) == set(missing_source["order_id"])

missing_clean.to_csv(
    MISSING_OUTPUT,
    index=False,
    encoding="utf-8",
    columns=EXPECTED_COLUMNS,
)

print(f"Created {MISSING_OUTPUT} with {len(missing_clean)} rows.")

Created Group018_missing_clean.csv with 500 rows.


## Task 2 - Missing-output audit and validation evidence

This section reads the exported file back from disk and records each validation check with its
observed result, pass/fail status and interpretation. The checks are independent readers of the
completed output; they do not feed values back into the cleaning calculations.

In [31]:
validation_results = []


def record_validation_result(
    results,
    check_id,
    check_name,
    observed_result,
    passed,
    interpretation,
):
    '''Append one executable validation result to a results list.

    Parameters
    ----------
    results : list of dict
        Mutable list collecting validation outcomes.
    check_id : str
        Short stable identifier used by audit evidence and notebook references.
    check_name : str
        Concise description of the condition being checked.
    observed_result : object
        Actual value or summary produced by the check.
    passed : bool
        Whether the observed result satisfies the requirement.
    interpretation : str
        Explanation of what the result establishes or how it was resolved.
    '''
    results.append(
        {
            "check_id": check_id,
            "check_name": check_name,
            "observed_result": str(observed_result),
            "status": "PASS" if bool(passed) else "FAIL",
            "interpretation": interpretation,
        }
    )

### 2.1 Read-back structure, completeness and preservation

This cell reads the submitted missing-data CSV from disk rather than validating only the in-memory
DataFrame. It checks schema, column order, row count, completeness and ID preservation. It also
compares every originally observed source cell with the corresponding output cell so valid values
cannot be silently changed.

In [32]:
missing_output_readback = pd.read_csv(
    MISSING_OUTPUT,
    dtype={"order_id": "string", "customer_id": "string"},
)

schema_passed = list(missing_output_readback.columns) == EXPECTED_COLUMNS
row_and_id_passed = (
    len(missing_output_readback) == EXPECTED_ROWS_PER_BATCH
    and missing_output_readback["order_id"].is_unique
    and set(missing_output_readback["order_id"]) == set(missing_source["order_id"])
)
completeness_passed = int(missing_output_readback.isna().sum().sum()) == 0

preservation_violations = []
source_by_id = missing_source.set_index("order_id")
output_by_id = missing_output_readback.set_index("order_id")

for column_name in EXPECTED_COLUMNS:
    if column_name == "order_id":
        continue
    observed_mask = source_by_id[column_name].notna()
    source_values = source_by_id.loc[observed_mask, column_name].astype(str)
    output_values = output_by_id.loc[observed_mask, column_name].astype(str)
    unequal_ids = source_values.index[source_values != output_values].tolist()
    preservation_violations.extend(
        (order_id, column_name) for order_id in unequal_ids
    )

preservation_passed = len(preservation_violations) == 0

record_validation_result(
    validation_results,
    "M01",
    "Exact schema and column order",
    list(missing_output_readback.columns),
    schema_passed,
    "The output must retain the published 16-column order.",
)
record_validation_result(
    validation_results,
    "M02",
    "Row count and preserved unique IDs",
    f"rows={len(missing_output_readback)}, unique_ids={missing_output_readback['order_id'].nunique()}",
    row_and_id_passed,
    "All 500 original missing-batch orders must remain present exactly once.",
)
record_validation_result(
    validation_results,
    "M03",
    "Required completeness",
    f"missing_cells={int(missing_output_readback.isna().sum().sum())}",
    completeness_passed,
    "Every omission in the missing batch must be recovered.",
)
record_validation_result(
    validation_results,
    "M04",
    "Preservation of originally observed values",
    f"violations={len(preservation_violations)}",
    preservation_passed,
    "Only source cells that were empty may differ in the cleaned output.",
)

### 2.2 Business-rule validation

This cell independently recalculates cart prices, seasons, warehouse distances, happiness and
order totals for every read-back row. It also applies the fitted seasonal delivery models to check
that the completed records follow the expected retained-row residual relationship.

In [33]:
cart_errors = []
price_errors = []
season_errors = []
warehouse_errors = []
distance_errors = []
happiness_errors = []
total_errors = []

for output_row in missing_output_readback.itertuples(index=False):
    try:
        parsed_cart = parse_shopping_cart(output_row.shopping_cart)
    except (TypeError, ValueError, json.JSONDecodeError) as error:
        cart_errors.append((output_row.order_id, str(error)))
        continue

    expected_price = calculate_order_price(parsed_cart, final_unit_prices)
    if abs(expected_price - output_row.order_price) > MONEY_TOLERANCE:
        price_errors.append(output_row.order_id)

    expected_season = get_australian_season(output_row.date)
    if expected_season != output_row.season:
        season_errors.append(output_row.order_id)

    expected_warehouse, expected_distance = find_nearest_warehouse(
        output_row.customer_lat,
        output_row.customer_long,
        warehouses_source,
    )
    if expected_warehouse != output_row.nearest_warehouse:
        warehouse_errors.append(output_row.order_id)
    if (
        abs(expected_distance - output_row.distance_to_nearest_warehouse)
        > DISTANCE_TOLERANCE
    ):
        distance_errors.append(output_row.order_id)

    expected_happiness = classify_customer_happiness(
        output_row.latest_customer_review,
        sentiment_analyser,
    )
    if expected_happiness != output_row.is_happy_customer:
        happiness_errors.append(output_row.order_id)

    expected_total = calculate_order_total(
        output_row.order_price,
        output_row.coupon_discount,
        output_row.delivery_charges,
    )
    if abs(expected_total - output_row.order_total) > MONEY_TOLERANCE:
        total_errors.append(output_row.order_id)

relationship_checks = [
    ("M05", "Valid shopping-cart structure", cart_errors),
    ("M06", "Cart-derived current order prices", price_errors),
    ("M07", "Calendar season relationship", season_errors),
    ("M08", "Nearest-warehouse relationship", warehouse_errors),
    ("M09", "Haversine-distance relationship", distance_errors),
    ("M10", "VADER happiness relationship", happiness_errors),
    ("M11", "Discount and order-total arithmetic", total_errors),
]

for check_id, check_name, errors in relationship_checks:
    record_validation_result(
        validation_results,
        check_id,
        check_name,
        f"violations={len(errors)}",
        len(errors) == 0,
        "The read-back values agree with the published rule and shared functions.",
    )

completed_model_residuals = []
for season_name in SEASON_ORDER:
    season_rows = missing_output_readback.loc[
        missing_output_readback["season"] == season_name
    ]
    predictions = predict_delivery_charges(
        season_rows,
        seasonal_delivery_models[season_name],
    )
    observed_charges = season_rows["delivery_charges"].to_numpy(dtype=float)
    completed_model_residuals.extend(
        np.abs(observed_charges - predictions).tolist()
    )

maximum_completed_residual = max(completed_model_residuals)
record_validation_result(
    validation_results,
    "M12",
    "Seasonal delivery-model residual behaviour",
    f"maximum_absolute_residual={maximum_completed_residual:.6f}",
    maximum_completed_residual <= 0.08,
    "Completed missing-batch rows remain within the delivery-charge tolerance.",
)

### 2.3 Difference and audit reconciliation

This cell derives the actual source-to-output differences and compares their order/field keys with
the 81 captured audit records. The check prevents missing audit entries, duplicate entries and audit
claims that do not correspond to a real output change.

In [34]:
actual_difference_keys = set()

for source_row in missing_source.itertuples(index=False):
    order_id = source_row.order_id
    for column_name in EXPECTED_COLUMNS:
        if column_name == "order_id":
            continue
        source_value = source_by_id.at[order_id, column_name]
        output_value = output_by_id.at[order_id, column_name]
        if pd.isna(source_value) and not pd.isna(output_value):
            actual_difference_keys.add((order_id, column_name))

audit_difference_keys = set(
    zip(
        missing_audit_records["record_key"],
        missing_audit_records["field_name"],
    )
)

audit_reconciliation_passed = (
    len(actual_difference_keys) == 81
    and len(audit_difference_keys) == 81
    and actual_difference_keys == audit_difference_keys
)

record_validation_result(
    validation_results,
    "M13",
    "Input/output differences agree with missing audit records",
    (
        f"actual_differences={len(actual_difference_keys)}, "
        f"audit_keys={len(audit_difference_keys)}"
    ),
    audit_reconciliation_passed,
    "Each recovered source omission has one matching Task 2 audit record.",
)

ending_source_hashes = {
    source_name: sha256_file(source_path)
    for source_name, source_path in source_paths.items()
}
source_hashes_passed = ending_source_hashes == initial_source_hashes
record_validation_result(
    validation_results,
    "M14",
    "Raw source files remain unchanged",
    f"unchanged={source_hashes_passed}",
    source_hashes_passed,
    "The workflow reads supplied inputs without overwriting them.",
)

### 2.4 Validation summary

This cell presents all missing-output checks in one concise table and stops execution if any check
fails. A passing table demonstrates structural correctness, business-rule consistency, preservation
of valid information and full agreement with the missing-data audit records.

In [35]:
missing_validation_results = pd.DataFrame(validation_results)
display(missing_validation_results)

failed_validation_checks = missing_validation_results.loc[
    missing_validation_results["status"] != "PASS"
]
assert failed_validation_checks.empty, failed_validation_checks.to_dict("records")

final_missing_summary = pd.Series(
    {
        "input_rows": len(missing_source),
        "output_rows": len(missing_output_readback),
        "source_missing_cells": int(original_missing_mask.sum().sum()),
        "output_missing_cells": int(missing_output_readback.isna().sum().sum()),
        "audit_records_prepared": len(missing_audit_records),
        "validation_checks_passed": int(
            (missing_validation_results["status"] == "PASS").sum()
        ),
        "validation_checks_failed": len(failed_validation_checks),
        "final_audit_log_written": False,
    },
    name="missing_task_result",
)
display(final_missing_summary)
print("Missing-value recovery and concurrent Task 2 validation: PASS")

,check_id,check_name,observed_result,status,interpretation
0,M01,Exact schema and column order,"['order_id', 'customer_id', 'date', 'nearest_w...",PASS,The output must retain the published 16-column...
1,M02,Row count and preserved unique IDs,"rows=500, unique_ids=500",PASS,All 500 original missing-batch orders must rem...
2,M03,Required completeness,missing_cells=0,PASS,Every omission in the missing batch must be re...
3,M04,Preservation of originally observed values,violations=0,PASS,Only source cells that were empty may differ i...
4,M05,Valid shopping-cart structure,violations=0,PASS,The read-back values agree with the published ...
5,M06,Cart-derived current order prices,violations=0,PASS,The read-back values agree with the published ...
6,M07,Calendar season relationship,violations=0,PASS,The read-back values agree with the published ...
7,M08,Nearest-warehouse relationship,violations=0,PASS,The read-back values agree with the published ...
8,M09,Haversine-distance relationship,violations=0,PASS,The read-back values agree with the published ...
9,M10,VADER happiness relationship,violations=0,PASS,The read-back values agree with the published ...


input_rows                    500
output_rows                   500
source_missing_cells           81
output_missing_cells            0
audit_records_prepared         81
validation_checks_passed       14
validation_checks_failed        0
final_audit_log_written     False
Name: missing_task_result, dtype: object

Missing-value recovery and concurrent Task 2 validation: PASS
